[Reference](https://medium.com/data-science-collective/what-is-vectorless-rag-the-three-stage-tree-and-reasoning-architecture-that-threw-out-embeddings-bdb8079af068$0)

```
root
└── Management Discussion and Analysis
    └── Results of Operations
        └── Quarterly Results
            └── Operating Income Table, pages 41-43
```

# 1. Build the Document Tree

In [1]:
import json
from pathlib import Path
from typing import Optional

from langchain_google_genai import ChatGoogleGenerativeAI
from pydantic import BaseModel, Field, ValidationError

ARCHITECT_MODEL = "gemini-2.5-flash"

class TreeNode(BaseModel):
    """A node in the document tree.
    Summaries route the search. The source of truth is the raw
    page range stored alongside the tree.
    """
    node_id: str = Field(pattern=r"^[a-z0-9_]+(\.[a-z0-9_]+)*$")
    title: str = Field(min_length=1, max_length=200)
    summary: str = Field(min_length=1, max_length=600)
    page_range: tuple[int, int]
    parent_path: list[str] = Field(default_factory=list)
    children: list[str] = Field(default_factory=list)
    has_table: bool = False
    source_refs: list[str] = Field(default_factory=list)

class ArchitectDecision(BaseModel):
    """What the LLM is allowed to return for one section."""
    title: str = Field(min_length=1, max_length=200)
    summary: str = Field(min_length=1, max_length=600)
    has_table: bool
    suggested_node_id: str = Field(pattern=r"^[a-z0-9_]+$")

_architect_llm = (
    ChatGoogleGenerativeAI(model=ARCHITECT_MODEL, temperature=0)
    .with_structured_output(ArchitectDecision)
)

_ARCHITECT_PROMPT = """You are indexing a structured document.
Given the raw text of one section, produce routing metadata only.

Rules:
- The summary describes WHAT THE READER WILL FIND in this section, in one
  sentence. It is metadata, not evidence. Do not paraphrase numbers.
- has_table is true only if the raw section contains a tabular layout
  (rows + columns of figures), not just a list.
- suggested_node_id is a short snake_case slug derived from the heading.

Heading: {heading}
Page range: {start}-{end}
Raw section text:
---
{raw}
---"""

def architect_node(
    heading: str,
    raw_text: str,
    page_range: tuple[int, int],
    parent_path: list[str],
) -> TreeNode:
    """Produce one validated TreeNode for one section."""
    prompt = _ARCHITECT_PROMPT.format(
        heading=heading, start=page_range[0], end=page_range[1], raw=raw_text[:8_000],
    )
    try:
        decision: ArchitectDecision = _architect_llm.invoke(prompt)
    except ValidationError as e:
        raise RuntimeError(f"architect_schema_invalid: {e.errors()[:1]}") from e

    node_id = ".".join(parent_path + [decision.suggested_node_id])
    return TreeNode(
        node_id=node_id,
        title=decision.title,
        summary=decision.summary,
        page_range=page_range,
        parent_path=parent_path,
        has_table=decision.has_table,
        source_refs=[f"{page_range[0]}-{page_range[1]}"],
    )

def build_tree(
    sections: list[dict],
    out_dir: Path,
) -> dict[str, TreeNode]:
    """Build the tree from already-parsed sections."""
    out_dir.mkdir(parents=True, exist_ok=True)
    raw_dir = out_dir / "raw"
    raw_dir.mkdir(exist_ok=True)

    tree: dict[str, TreeNode] = {}
    for section in sections:
        node = architect_node(
            heading=section["heading"],
            raw_text=section["raw_text"],
            page_range=tuple(section["page_range"]),
            parent_path=section["parent_path"],
        )
        tree[node.node_id] = node
        (raw_dir / f"{node.node_id}.txt").write_text(section["raw_text"])

    for node in tree.values():
        if node.parent_path:
            parent_id = ".".join(node.parent_path)
            if parent_id in tree:
                tree[parent_id].children.append(node.node_id)

    (out_dir / "tree.json").write_text(
        json.dumps({nid: n.model_dump() for nid, n in tree.items()}, indent=2)
    )
    return tree

def load_raw_section(out_dir: Path, node_id: str) -> Optional[str]:
    """Source of truth at answer time. Never read from the summary."""
    p = out_dir / "raw" / f"{node_id}.txt"
    return p.read_text() if p.exists() else None

# 2. Traverse with Reasoning


In [2]:

import json
import time
from pathlib import Path
from typing import Annotated, Literal, TypedDict

from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import START, END, StateGraph
from pydantic import BaseModel, Field

from snippet1 import TreeNode

REASONER_MODEL = "gemini-2.5-pro"
DEFAULT_BUDGET = 8
TRACE_PATH = Path("traces/traversal.jsonl")
TRACE_PATH.parent.mkdir(exist_ok=True)

class CandidateView(BaseModel):
    """What the reasoner is allowed to see for each candidate node.
    Raw section text is intentionally excluded.
    """
    node_id: str
    title: str
    summary: str
    page_range: tuple[int, int]
    has_table: bool
    parent_path: list[str]

class TraversalDecision(BaseModel):
    select_for_evidence: list[str] = Field(default_factory=list)
    expand_next: list[str] = Field(default_factory=list)
    reason: str = Field(max_length=400)
    has_enough_evidence: bool
    evidence_gap: str = Field(default="", max_length=400)

def _merge_unique(
    existing: list[str], new: list[str]
) -> list[str]:
    seen = set(existing)
    return existing + [x for x in new if not (x in seen or seen.add(x))]

class TraversalState(TypedDict):
    query: str
    tree: dict[str, TreeNode]
    frontier: list[str]
    selected: Annotated[list[str], _merge_unique]
    evidence_gap: str
    budget_left: int
    done: bool

_reasoner_llm = (
    ChatGoogleGenerativeAI(model=REASONER_MODEL, temperature=0)
    .with_structured_output(TraversalDecision)
)

_REASONER_PROMPT = """You are exploring a structured document to answer a query.
You see candidate nodes (title, summary, page range, parent path) — NOT raw text.
Your job is to (a) select nodes that look likely to contain the literal evidence
and (b) decide which child branches deserve to be expanded next.

Query: {query}
Current evidence gap: {gap}
Budget remaining (steps): {budget}
Candidates this step:
{candidates}

Rules:
- Prefer leaf nodes for select_for_evidence. Expand internal nodes via expand_next.
- has_enough_evidence is true only if the selected nodes plausibly contain
  every fact the query asks for, including comparison periods or exclusions.
- evidence_gap describes what is still missing in plain language."""

def _candidate_views(
    state: TraversalState,
) -> list[CandidateView]:
    return [
        CandidateView(**state["tree"][nid].model_dump())
        for nid in state["frontier"]
        if nid in state["tree"]
    ]

def _trace(turn: int, state: TraversalState, decision: TraversalDecision) -> None:
    rec = {
        "ts": time.time(),
        "turn": turn,
        "query": state["query"],
        "frontier": state["frontier"],
        "candidates": [c.node_id for c in _candidate_views(state)],
        "selected": decision.select_for_evidence,
        "expand_next": decision.expand_next,
        "reason": decision.reason,
        "evidence_gap": decision.evidence_gap,
        "budget_left": state["budget_left"],
    }
    with TRACE_PATH.open("a") as f:
        f.write(json.dumps(rec) + "\n")

def reason_step(state: TraversalState) -> dict:
    candidates = _candidate_views(state)
    if not candidates:
        return {"done": True, "evidence_gap": "frontier_empty"}

    prompt = _REASONER_PROMPT.format(
        query=state["query"],
        gap=state["evidence_gap"] or "(none yet)",
        budget=state["budget_left"],
        candidates="\n".join(c.model_dump_json() for c in candidates),
    )
    decision: TraversalDecision = _reasoner_llm.invoke(prompt)

    turn = DEFAULT_BUDGET - state["budget_left"] + 1
    _trace(turn, state, decision)

    next_frontier: list[str] = []
    for nid in decision.expand_next:
        node = state["tree"].get(nid)
        if node is not None:
            next_frontier.extend(node.children)

    return {
        "selected": decision.select_for_evidence,
        "frontier": next_frontier,
        "evidence_gap": decision.evidence_gap,
        "budget_left": state["budget_left"] - 1,
        "done": decision.has_enough_evidence,
    }

def route_after_reason(
    state: TraversalState,
) -> Literal["reason", "end"]:
    if state["done"]:
        return "end"
    if state["budget_left"] <= 0:
        return "end"
    if not state["frontier"]:
        return "end"
    return "reason"

graph = StateGraph(TraversalState)
graph.add_node("reason", reason_step)
graph.add_edge(START, "reason")
graph.add_conditional_edges(
    "reason", route_after_reason, {"reason": "reason", "end": END}
)
traverse = graph.compile()

def run_traversal(
    query: str, tree: dict[str, TreeNode], root_id: str
) -> TraversalState:
    initial: TraversalState = {
        "query": query,
        "tree": tree,
        "frontier": [root_id, *tree[root_id].children],
        "selected": [],
        "evidence_gap": "",
        "budget_left": DEFAULT_BUDGET,
        "done": False,
    }
    return traverse.invoke(initial)

# 3. Answer from Evidence

In [3]:

import re
from pathlib import Path
from typing import Literal

from langchain_google_genai import ChatGoogleGenerativeAI
from pydantic import BaseModel, Field

from snippet1 import TreeNode, load_raw_section

ANSWER_MODEL = "gemini-2.5-pro"

class Citation(BaseModel):
    node_id: str
    page: int
    quote: str = Field(min_length=1, max_length=400)

class GroundedAnswer(BaseModel):
    answer: str = Field(min_length=1, max_length=2_000)
    calculation: str = Field(default="", max_length=600)
    citations: list[Citation]
    confidence: Literal["high", "medium", "low"] = "medium"
    missing_evidence: list[str] = Field(default_factory=list)

class VerifyResult(BaseModel):
    ok: bool
    failures: list[str] = Field(default_factory=list)

_answer_llm = (
    ChatGoogleGenerativeAI(model=ANSWER_MODEL, temperature=0)
    .with_structured_output(GroundedAnswer)
)

_ANSWER_PROMPT = """Answer the query using ONLY the source sections below.
Every numeric claim must be backed by a citation whose `quote` is a literal
substring of the cited section. If the sources do not contain the answer
(including any requested exclusions or comparison periods), set confidence to
"low" and list the missing facts in `missing_evidence`. Do not infer values
that are not in the source text.

Query: {query}

Source sections:
{sources}"""

def _format_sources(
    selected: list[str], tree: dict[str, TreeNode], out_dir: Path
) -> tuple[str, dict[str, str]]:
    blocks: list[str] = []
    raw_by_id: dict[str, str] = {}
    for nid in selected:
        node = tree.get(nid)
        raw = load_raw_section(out_dir, nid) if node else None
        if not node or raw is None:
            continue
        raw_by_id[nid] = raw
        header = (
            f"[{nid}] {node.title} (pages {node.page_range[0]}-{node.page_range[1]})"
        )
        blocks.append(f"{header}\n{raw}")
    return "\n\n".join(blocks), raw_by_id

_NUMBER = re.compile(r"-?\$?\d[\d,]*(?:\.\d+)?%?")
_EXCLUSION_HINTS = ("excluding", "ex-", "before", "without", "net of")

def verify(
    answer: GroundedAnswer,
    query: str,
    selected: list[str],
    tree: dict[str, TreeNode],
    raw_by_id: dict[str, str],
) -> VerifyResult:
    failures: list[str] = []

    selected_set = set(selected)
    for c in answer.citations:
        if c.node_id not in selected_set:
            failures.append(f"citation_node_not_selected:{c.node_id}")
            continue
        node = tree.get(c.node_id)
        if node is None:
            failures.append(f"citation_node_missing:{c.node_id}")
            continue
        lo, hi = node.page_range
        if not (lo <= c.page <= hi):
            failures.append(
                f"citation_page_outside_range:{c.node_id}:{c.page} not in {lo}-{hi}"
            )
        raw = raw_by_id.get(c.node_id, "")
        if c.quote.strip() and c.quote.strip() not in raw:
            failures.append(f"citation_quote_not_in_source:{c.node_id}")

    answer_numbers = set(_NUMBER.findall(answer.answer))
    cited_numbers: set[str] = set()
    for c in answer.citations:
        cited_numbers.update(_NUMBER.findall(c.quote))
    ungrounded = answer_numbers - cited_numbers
    if ungrounded:
        failures.append(f"ungrounded_numeric_claims:{sorted(ungrounded)[:3]}")

    if any(h in query.lower() for h in _EXCLUSION_HINTS):
        if not any(
            h in (answer.answer + answer.calculation).lower()
            for h in _EXCLUSION_HINTS
        ):
            failures.append("exclusion_clause_not_addressed")

    return VerifyResult(ok=not failures, failures=failures)

REFUSAL_TEMPLATE = (
    "Not enough evidence in the retrieved sections to answer with confidence. "
    "Verifier failures: {failures}. Missing facts: {missing}."
)

def answer_from_evidence(
    query: str,
    selected: list[str],
    tree: dict[str, TreeNode],
    out_dir: Path,
) -> GroundedAnswer:
    if not selected:
        return GroundedAnswer(
            answer=REFUSAL_TEMPLATE.format(
                failures=["no_selected_nodes"], missing=["all"]
            ),
            citations=[],
            confidence="low",
            missing_evidence=["traversal_returned_empty"],
        )

    sources_text, raw_by_id = _format_sources(selected, tree, out_dir)
    if not sources_text:
        return GroundedAnswer(
            answer=REFUSAL_TEMPLATE.format(
                failures=["raw_text_unavailable"], missing=selected
            ),
            citations=[],
            confidence="low",
            missing_evidence=["raw_sections_missing"],
        )

    candidate: GroundedAnswer = _answer_llm.invoke(
        _ANSWER_PROMPT.format(query=query, sources=sources_text)
    )
    result = verify(candidate, query, selected, tree, raw_by_id)
    if result.ok:
        return candidate

    return GroundedAnswer(
        answer=REFUSAL_TEMPLATE.format(
            failures=result.failures, missing=candidate.missing_evidence or ["unknown"]
        ),
        calculation="",
        citations=candidate.citations,
        confidence="low",
        missing_evidence=candidate.missing_evidence + result.failures,
    )